---
execute:
    echo: false
---

# Elite Dangerous Database Reader
> Read Elite Dangerous database files

In [ ]:
#| default_exp eddb.readers

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export

import sys, time, logging, json, gzip, os, io
import pandas as pd
import edcompanion.core

from pathlib import Path
from edcompanion.core import configuration


In [ ]:
from confproxy.core import init_console_logging


In [ ]:
init_console_logging(__name__)

2025-12-20T19:59:08+0100 INFO	3173	__main__	core.py	init_console_logging	42	There already is a logger installed for __main__.


<Logger __main__ (INFO)>

In [ ]:
#| exporti
syslog = logging.getLogger(__name__)
syslog.info(f"Loading module {__name__}")

2025-12-20T19:59:08+0100 INFO	3173	__main__	2923348250.py	<module>	3	Loading module __main__


In [ ]:
eddb_conf = configuration['EDDB']

### File Reader as generator

In [ ]:
#| export
def dbfilereader(filename):
    """
        Opens 'filename' as generator for eddb style objects
    """

    chunksize = 64 * 1024 * 1024

    with gzip.open(filename, 'rt') as jsonfile:

        while True:
            chunk = jsonfile.readlines(chunksize)
            if chunk:
                for line in chunk:
                    if len(line) < 5:
                        continue

                    yield json.loads(line.rstrip(',\n\r '))

            else:
                break




In [ ]:
eddb_conf['systems_1day'] = "systems_1day.json.gz"

In [ ]:
systems1 = os.path.join(eddb_conf['local_dumps'], eddb_conf['systems_1day'])
print(systems1)

/home/fenke/repos/EDCompanion/data/systems_1day.json.gz


In [ ]:
os.path.exists("~/repos/EDCompanion/data/systems_1day.json.gz")

False

In [ ]:
i = 0
for item in dbfilereader(systems1):
    print(item)
    i+=1
    if i > 10:
        break

{'id64': 2326687, 'name': 'HD 192281', 'mainStar': 'O (Blue-White) Star', 'coords': {'x': -4023.53125, 'y': 230.875, 'z': 896.46875}, 'updateTime': '2024-04-13 21:57:30+00'}
{'id64': 2587943, 'name': 'Great Annihilator', 'mainStar': 'Black Hole', 'coords': {'x': 354.84375, 'y': -42.4375, 'z': 22997.21875}, 'updateTime': '2024-04-13 09:33:09+00'}
{'id64': 10451223, 'name': 'Spoihaae AA-A h1', 'mainStar': 'O (Blue-White) Star', 'coords': {'x': -9578.15625, 'y': -1438.1875, 'z': 20029.15625}, 'updateTime': '2024-04-13 19:00:44+00'}
{'id64': 11926022, 'name': 'Eorgh Flyiae ZE-A g0', 'mainStar': 'O (Blue-White) Star', 'coords': {'x': -20675.03125, 'y': -470.875, 'z': 16966.21875}, 'updateTime': '2024-04-13 10:19:43+00'}
{'id64': 17955166, 'name': 'Drojao CL-Y g0', 'mainStar': 'B (Blue-White) Star', 'coords': {'x': -6316.5, 'y': -394.3125, 'z': 3815.78125}, 'updateTime': '2024-04-13 18:15:19+00'}
{'id64': 18743542, 'name': 'MJD95 J023559.03+611735.8', 'mainStar': 'O (Blue-White) Star', 'coor

### File Reader as chunked processor

In [ ]:
#| export

def dbfile_process(filename, process_chunk):
    """Opens file and calls process_chunk to process batches of items"""

    chunksize = 16 * 1024 * 1024

    with gzip.open(filename, 'rt') as jsonfile:

        while True:
            chunk = jsonfile.readlines(chunksize)
            if chunk:
                data = []
                for line in chunk:
                    if len(line) < 5:
                        continue

                    item = json.loads(line.rstrip(',\n\r '))
                    data.append(item)

                process_chunk(data)

            else:
                break



### File reader using pandas

In [ ]:
? gzip

Type:        module
String form: <module 'gzip' from '/usr/lib64/python3.11/gzip.py'>
File:        /usr/lib64/python3.11/gzip.py
Docstring:  
Functions that read and write gzipped files.

The user of the file doesn't have to worry about the compression,
but random access is not allowed.

In [ ]:
? gzip.open

Signature:
 gzip.open(
    filename,
    mode='rb',
    compresslevel=9,
    encoding=None,
    errors=None,
    newline=None,
)
Docstring:
Open a gzip-compressed file in binary or text mode.

The filename argument can be an actual filename (a str or bytes object), or
an existing file object to read from or write to.

The mode argument can be "r", "rb", "w", "wb", "x", "xb", "a" or "ab" for
binary mode, or "rt", "wt", "xt" or "at" for text mode. The default mode is
"rb", and the default compresslevel is 9.

For binary mode, this function is equivalent to the GzipFile constructor:
GzipFile(filename, mode, compresslevel). In this case, the encoding, errors
and newline arguments must not be provided.

For text mode, a GzipFile object is created, and wrapped in an
io.TextIOWrapper instance with the specified encoding, error handling
behavior, and line ending(s).
File:      /usr/lib64/python3.11/gzip.py
Type:      function

In [ ]:
? io.TextIOWrapper.readlines

Signature:  io.TextIOWrapper.readlines(self, hint=-1, /)
Docstring:
Return a list of lines from the stream.

hint can be specified to control the number of lines read: no more
lines will be read if the total size (in bytes/characters) of all
lines so far exceeds hint.
Type:      method_descriptor

In [ ]:
#| export

def dbfile_process_dataframes(filename, process_chunk):
    """Opens file and calls process_chunk to process batches of items"""

    chunksize = 128 * 1024

    with gzip.open(filename, 'rt') as jsonfile:

        while True:
            chunk = jsonfile.readlines(chunksize)
            if chunk:

                df = pd.read_json()

                process_chunk(data)

            else:
                break



### File Reader as async chunked processor

In [ ]:
#| export

async def dbfile_process_async(filename, process_chunk):
    """Opens file and calls process_chunk to process batches of items"""

    chunksize = 16 * 1024 * 1024

    with gzip.open(filename, 'rt') as jsonfile:

        while True:
            chunk = jsonfile.readlines(chunksize)
            if chunk:
                data = []
                for line in chunk:
                    if len(line) < 5:
                        continue

                    item = json.loads(line.rstrip(',\n\r '))
                    data.append(item)

                await process_chunk(data)

            else:
                break



### File Reader through pandas

In [ ]:
def 

In [ ]:
#| hidey
import nbdev; nbdev.nbdev_export()